# Lab: Curriculum reward cho Sum-to-Three

Notebook hướng dẫn này dành cho sinh viên đã biết các khái niệm RL cơ bản. Mục tiêu là **đọc và phản biện bằng chứng** của T0/T1/T2, không chạy huấn luyện dài. Chỉ dùng thư viện chuẩn, NumPy, Matplotlib và (nếu có) TensorBoard EventAccumulator.

**Luồng làm việc:** tạo manifest → nạp JSON/TensorBoard → kiểm tra random policy → vẽ learning curve → thống kê theo seed → kiểm tra reward hacking → ghi journal và ra quyết định go/no-go.

## Protocol tối thiểu

- **Arms A–D:** factorial reward (`binary`/`event_aligned`) × starts (`canonical`/curriculum).
- **Arm E:** dense reward + broad starts ngay từ đầu, kiểm tra curriculum ordering.
- **Test distributions:** T0 canonical, T1 local perturbation, T2 broad valid random.
- **Primary evidence:** binary episode score và binary per-shot success; shaped collector return chỉ là diagnostic.

Notebook đọc tracker export, TensorBoard export, checkpoint evaluation JSON và random-policy diagnostic do các script trong `../scripts/` tạo ra. Nó không chạy training dài.

In [ ]:
from __future__ import annotations

import json
import math
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ImportError:
    plt = None
    MATPLOTLIB_AVAILABLE = False

RNG = np.random.default_rng(2026)
if MATPLOTLIB_AVAILABLE:
    plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    TENSORBOARD_AVAILABLE = True
except ImportError:
    EventAccumulator = None
    TENSORBOARD_AVAILABLE = False

matplotlib_status = plt.matplotlib.__version__ if MATPLOTLIB_AVAILABLE else 'missing (required for plots)'
tensorboard_status = 'available' if TENSORBOARD_AVAILABLE else 'missing (optional)'
print(f'Python: {sys.version.split()[0]} | NumPy: {np.__version__}')
print(f'Matplotlib: {matplotlib_status} | TensorBoard EventAccumulator: {tensorboard_status}')
print(f'python executable: {shutil.which("python") or sys.executable}')
if not MATPLOTLIB_AVAILABLE:
    print('Cài để vẽ: python -m pip install matplotlib')
if not TENSORBOARD_AVAILABLE:
    print('Cài tùy chọn khi cần đọc event files: python -m pip install tensorboard')

## 1. Tạo run ID, command và manifest

Không chạy command trong notebook này. Cell sau chỉ tạo command có thể copy vào terminal. Giữ `config`, `seed`, `experiment_dir` và đường dẫn metrics để phép so sánh tái lập được.

In [ ]:
ARMS = (
    'A_sparse_fixed', 'B_dense_fixed', 'C_sparse_curriculum',
    'D_dense_curriculum', 'E_dense_broad',
)
TEST_DISTRIBUTIONS = ('canonical', 'local', 'broad')


def make_run_id(arm: str, seed: int, stage: str = 'main') -> str:
    if arm not in ARMS:
        raise ValueError(f'arm phải là một trong {ARMS}')
    return f'{arm}-seed-{int(seed)}-{stage}'


def make_command(arm: str, seed: int, max_env_step: int = 40_000, cpu: bool = False) -> str:
    if arm not in ARMS:
        raise ValueError(f'arm phải là một trong {ARMS}')
    suffix = ' --cpu' if cpu else ''
    return (
        'python zoo/pooltool/sum_to_three/config/research_alpha.py '
        f'--arm {arm} --seed {int(seed)} --max-env-step {int(max_env_step)}{suffix}'
    )


def new_run(arm: str, seed: int, max_env_step: int = 40_000, status: str = 'planned') -> dict:
    if status not in {'planned', 'running', 'done', 'blocked'}:
        raise ValueError('status không hợp lệ')
    return {
        'run_id': make_run_id(arm, seed),
        'arm': arm,
        'seed': int(seed),
        'status': status,
        'max_env_step': int(max_env_step),
        'command': make_command(arm, seed, max_env_step),
        'metrics': [],
        'evaluations': {},
        'notes': '',
    }

example_run = new_run('D_dense_curriculum', 0, max_env_step=2_000)
print(json.dumps(example_run, ensure_ascii=False, indent=2))

## 2. Nạp normalized JSON

Chọn file JSON được xuất từ tracker hoặc một file chứa `runs`. Ngoài các khóa manifest, notebook giữ nguyên trường bổ sung như `metrics` và `random_policy`. Một điểm metric có dạng ví dụ: `{"step": 2000, "eval_return": 0.71, "true_success_rate": 0.55}`.

In [ ]:
def _series_to_metrics(payload: dict) -> list[dict]:
    tag = payload.get('primary_binary_series_tag')
    events = payload.get('series', {}).get(tag, []) if tag else []
    return [
        {'step': int(event['step']), 'binary_eval_return': float(event['value'])}
        for event in events
    ]


def normalize_run(record: dict) -> dict:
    if not isinstance(record, dict):
        raise TypeError('mỗi run phải là một object JSON')

    experiment = record.get('experiment', {}) if isinstance(record.get('experiment'), dict) else {}
    reproducibility = record.get('reproducibility', {}) if isinstance(record.get('reproducibility'), dict) else {}
    seeds = reproducibility.get('seeds', []) if isinstance(reproducibility, dict) else []
    arm = record.get('arm') or experiment.get('family') or ''
    seed = record.get('seed', seeds[0] if isinstance(seeds, list) and seeds else None)
    run = {
        'run_id': record.get('run_id', ''),
        'arm': arm,
        'seed': int(seed) if seed not in ('', None) else None,
        'status': record.get('status', ''),
        'metrics': record.get('metrics', []),
        'evaluations': record.get('evaluations', {}),
        'source': record,
    }

    if record.get('series'):
        run['metrics'] = _series_to_metrics(record)
    if isinstance(record.get('evaluation'), dict):
        distribution = record['evaluation'].get('test_distribution', 'canonical')
        run['evaluations'] = {distribution: record['evaluation']}
    return run


def load_normalized_json(path: str | Path) -> tuple[dict, list[dict]]:
    path = Path(path)
    with path.open(encoding='utf-8') as handle:
        raw = json.load(handle)
    if isinstance(raw, list):
        envelope, records = {'schema_version': 1}, raw
    elif isinstance(raw, dict) and isinstance(raw.get('runs'), list):
        envelope, records = raw, raw['runs']
    elif isinstance(raw, dict) and raw.get('run_id'):
        envelope, records = {'schema_version': raw.get('schema_version', 1)}, [raw]
    else:
        raise ValueError('JSON cần là tracker export, TensorBoard export, evaluation JSON, hoặc run manifest.')
    runs = [normalize_run(record) for record in records]
    return envelope, runs


# Có thể nạp nhiều file rồi nối runs, ví dụ tracker + TensorBoard exports + evaluations.
JSON_PATHS: list[Path] = []
runs = []
for path in JSON_PATHS:
    envelope, loaded = load_normalized_json(path)
    runs.extend(loaded)
    print(f'Đã nạp {len(loaded)} records từ {path}')
if not JSON_PATHS:
    envelope = {'schema_version': 1}
    print('Chưa chọn JSON_PATHS; cell kế tiếp tạo dữ liệu tổng hợp để test notebook.')

In [ ]:
def merge_run_records(records: list[dict]) -> list[dict]:
    """Gộp artifacts chỉ khi cùng exact run_id và không xung đột checkpoint/provenance."""
    merged = {}
    for record in records:
        run_id = record.get('run_id')
        if not run_id or run_id == 'unknown':
            raise ValueError('Mọi artifact phải có exact run_id, gồm attempt label.')
        target = merged.setdefault(run_id, {
            'run_id': run_id, 'arm': record.get('arm'), 'seed': record.get('seed'),
            'status': record.get('status', ''), 'metrics': [], 'evaluations': {},
            'checkpoint_sha256': None, 'source_records': [],
        })
        if target['arm'] not in (None, '', record.get('arm')) or target['seed'] not in (None, record.get('seed')):
            raise ValueError(f'Conflicting arm/seed for {run_id}')
        source = record.get('source', {})
        checkpoint_sha = record.get('checkpoint_sha256') or source.get('checkpoint_sha256')
        if checkpoint_sha and target['checkpoint_sha256'] not in (None, checkpoint_sha):
            raise ValueError(f'Multiple checkpoint hashes for {run_id}')
        if checkpoint_sha:
            target['checkpoint_sha256'] = checkpoint_sha
        if record.get('metrics'):
            if target['metrics'] and target['metrics'] != record['metrics']:
                raise ValueError(f'Multiple metric series for {run_id}')
            target['metrics'] = record['metrics']
        for distribution, evaluation in record.get('evaluations', {}).items():
            if distribution in target['evaluations'] and target['evaluations'][distribution] != evaluation:
                raise ValueError(f'Conflicting {distribution} evaluation for {run_id}')
            target['evaluations'][distribution] = evaluation
        target['source_records'].append(source)
    return list(merged.values())


def make_demo_runs() -> list[dict]:
    """Synthetic data để test pipeline — KHÔNG phải kết quả nghiên cứu."""
    demo = []
    steps = np.array([0, 2_000, 5_000, 10_000, 20_000, 40_000], dtype=float)
    gains = {
        'A_sparse_fixed': 3.4, 'B_dense_fixed': 4.2,
        'C_sparse_curriculum': 4.6, 'D_dense_curriculum': 5.4,
    }
    for arm, gain in gains.items():
        for seed in (0, 1, 2):
            run = new_run(arm, seed, status='done')
            noise = np.random.default_rng(seed + len(arm)).normal(0, 0.18, len(steps))
            binary_return = np.clip(0.2 + gain * (1 - np.exp(-steps / 12_000)) + noise, 0, 10)
            collector_return = np.clip(binary_return + (0.8 if arm in {'B_dense_fixed', 'D_dense_curriculum'} else 0), 0, 10)
            run['metrics'] = [
                {'step': int(step), 'binary_eval_return': float(binary_value), 'collector_return': float(collector_value)}
                for step, binary_value, collector_value in zip(steps, binary_return, collector_return)
            ]
            run['evaluations'] = {
                distribution: {
                    'test_distribution': distribution,
                    'binary_episode_score_mean': float(binary_return[-1] * factor),
                    'binary_per_shot_success_rate': float(binary_return[-1] * factor / 10),
                }
                for distribution, factor in {'canonical': 1.0, 'local': 0.86, 'broad': 0.68}.items()
            }
            demo.append(run)
    return demo

if not runs:
    runs = make_demo_runs()
    print(f'Dùng {len(runs)} synthetic runs. Thay JSON_PATHS để phân tích dữ liệu thật.')
else:
    runs = merge_run_records(runs)
    print(f'Đã gộp thành {len(runs)} exact run records.')

## 3. Nạp TensorBoard scalar (tùy chọn)

Nếu JSON chưa chứa `metrics`, dùng hàm này để đọc event file. Tag khác nhau giữa project; in `available_tags` trước, rồi truyền mapping rõ ràng. Không lỗi nếu TensorBoard chưa được cài.

In [ ]:
def load_tensorboard_scalars(event_path: str | Path, tag_map: dict[str, str] | None = None) -> tuple[dict[str, list[dict]], list[str]]:
    """Trả về metric normalized và toàn bộ scalar tags."""
    if not TENSORBOARD_AVAILABLE:
        print('TensorBoard không có; bỏ qua. Cài: python -m pip install tensorboard')
        return {}, []
    accumulator = EventAccumulator(str(event_path), size_guidance={'scalars': 0})
    accumulator.Reload()
    available_tags = accumulator.Tags().get('scalars', [])
    tag_map = tag_map or {}  # {'eval_return': 'your/tag', 'true_success_rate': 'your/other/tag'}
    series = {}
    for metric_name, tag in tag_map.items():
        if tag not in available_tags:
            print(f'Không thấy tag {tag!r}; bỏ qua {metric_name}.')
            continue
        series[metric_name] = [{'step': event.step, metric_name: float(event.value), 'wall_time': event.wall_time}
                               for event in accumulator.Scalars(tag)]
    return series, available_tags

# event_scalars, tags = load_tensorboard_scalars('path/to/events.out.tfevents...', {'eval_return': '...'} )
# print(tags)

## 4. Random-policy diagnostics

Nạp JSON do `../scripts/random_policy_diagnostics.py` tạo. Mục tiêu là đo `contact_rate`, `exact_three_success_rate` và cushion histogram trên canonical/local/broad trước khi gọi task là sparse hoặc khó.

In [ ]:
def random_policy_diagnostics(payload: dict) -> list[dict]:
    results = payload.get('results', []) if isinstance(payload, dict) else []
    if not results:
        raise ValueError('Cần JSON có key results từ random_policy_diagnostics.py')
    rows = []
    for result in results:
        row = {
            'distribution': result['start_distribution'],
            'episodes': int(result['episodes']),
            'shots': int(result['shots']),
            'contact_rate': float(result['contact_rate']),
            'exact_three_success_rate': float(result['exact_three_success_rate']),
            'cushion_histogram_given_contact': result.get('cushion_histogram_given_contact', {}),
        }
        rows.append(row)
        print(
            f"{row['distribution']:>9} | episodes={row['episodes']} | "
            f"contact={row['contact_rate']:.3f} | exact3={row['exact_three_success_rate']:.4f} | "
            f"hist={row['cushion_histogram_given_contact']}"
        )
    return rows

RANDOM_DIAGNOSTIC_PATH: Path | None = None
if RANDOM_DIAGNOSTIC_PATH:
    random_payload = json.loads(RANDOM_DIAGNOSTIC_PATH.read_text(encoding='utf-8'))
    random_summary = random_policy_diagnostics(random_payload)
else:
    random_summary = []
    print('Đặt RANDOM_DIAGNOSTIC_PATH để nạp output thật; không tạo fake diagnostic.')

## 5. Binary learning curves

Vẽ `binary_eval_return` theo environment step cho từng arm. Vùng mờ là min–max giữa seeds, không phải confidence interval. `collector_return` có thể vẽ riêng để chẩn đoán nhưng không thay headline metric.

In [ ]:
ARM_STYLE = {
    'A_sparse_fixed': {'color': '#1D4ED8', 'marker': 'o'},
    'B_dense_fixed': {'color': '#C2410C', 'marker': 's'},
    'C_sparse_curriculum': {'color': '#087F5B', 'marker': 'D'},
    'D_dense_curriculum': {'color': '#A21CAF', 'marker': '^'},
    'E_dense_broad': {'color': '#5D687A', 'marker': 'v'},
}


def metric_points(run: dict, metric: str) -> tuple[np.ndarray, np.ndarray]:
    rows = [
        row for row in run.get('metrics', [])
        if isinstance(row, dict) and row.get('step') is not None and row.get(metric) is not None
    ]
    rows.sort(key=lambda row: float(row['step']))
    return (
        np.asarray([float(row['step']) for row in rows]),
        np.asarray([float(row[metric]) for row in rows]),
    )


def plot_learning_curves(runs: list[dict], metric: str = 'binary_eval_return', points: int = 100):
    if not MATPLOTLIB_AVAILABLE:
        print('Thiếu matplotlib. Cài: python -m pip install matplotlib')
        return None, None
    fig, ax = plt.subplots(figsize=(9.5, 5), layout='constrained')
    drawn = 0
    for arm in ARMS:
        series = [metric_points(run, metric) for run in runs if run.get('arm') == arm]
        series = [(x, y) for x, y in series if len(x) >= 2]
        if not series:
            continue
        start, end = max(x[0] for x, _ in series), min(x[-1] for x, _ in series)
        if end <= start:
            continue
        grid = np.linspace(start, end, points)
        values = np.vstack([np.interp(grid, x, y) for x, y in series])
        style = ARM_STYLE[arm]
        ax.plot(
            grid, values.mean(axis=0), color=style['color'], marker=style['marker'],
            markevery=max(1, points // 6), lw=2, ms=5, label=f'{arm} (n={len(series)})',
        )
        ax.fill_between(grid, values.min(axis=0), values.max(axis=0), color=style['color'], alpha=.12)
        drawn += 1
    ax.set(title=f'Learning curve — {metric}', xlabel='Environment steps', ylabel=metric)
    ax.grid(axis='y', color='#D8E0EC', linewidth=.8)
    if drawn:
        ax.legend(frameon=False, fontsize=8)
    else:
        ax.text(.5, .5, f'Không có metric {metric!r}', ha='center', va='center', transform=ax.transAxes)
    return fig, ax

fig, ax = plot_learning_curves(runs)
if fig is not None:
    plt.show()

## 6. AUC, bảng per-seed và bootstrap CI

AUC được chuẩn hóa theo độ rộng step chung của **mỗi run**, vì vậy nó nằm cùng đơn vị metric và trả lời "một run học tốt trung bình trong budget của chính nó không?". Khi budget khác nhau, đừng dùng AUC để kết luận trực tiếp; hãy lọc hoặc tái chạy với budget bằng nhau. Bootstrap ở đây là CI mô tả trên seed, không thay thế thiết kế thực nghiệm.

In [ ]:
def final_metric(run: dict, metric: str) -> float:
    _, y = metric_points(run, metric)
    return float(y[-1]) if len(y) else math.nan


def normalized_auc(run: dict, metric: str) -> float:
    x, y = metric_points(run, metric)
    if len(x) < 2 or x[-1] <= x[0]:
        return math.nan
    area = np.trapezoid(y, x) if hasattr(np, 'trapezoid') else np.trapz(y, x)
    return float(area / (x[-1] - x[0]))


def per_seed_table(runs: list[dict], metric: str = 'binary_eval_return') -> list[dict]:
    rows = [
        {
            'arm': run.get('arm'),
            'seed': run.get('seed'),
            'run_id': run.get('run_id'),
            f'final_{metric}': final_metric(run, metric),
            f'auc_{metric}': normalized_auc(run, metric),
        }
        for run in runs
    ]
    return sorted(rows, key=lambda row: (str(row['arm']), -1 if row['seed'] is None else row['seed']))


def print_table(rows: list[dict], digits: int = 3) -> None:
    if not rows:
        print('(trống)')
        return
    columns = list(rows[0])
    text_rows = [
        [f'{value:.{digits}f}' if isinstance(value, float) and np.isfinite(value) else str(value)
         for value in [row.get(column, '') for column in columns]]
        for row in rows
    ]
    widths = [max(len(column), *(len(row[i]) for row in text_rows)) for i, column in enumerate(columns)]
    print(' | '.join(column.ljust(widths[i]) for i, column in enumerate(columns)))
    print('-+-'.join('-' * width for width in widths))
    for row in text_rows:
        print(' | '.join(value.ljust(widths[i]) for i, value in enumerate(row)))


def bootstrap_ci(values, n_boot: int = 10_000, confidence: float = .95, rng=None) -> dict:
    values = np.asarray([value for value in values if np.isfinite(value)], dtype=float)
    if len(values) == 0:
        return {'n': 0, 'mean': math.nan, 'low': math.nan, 'high': math.nan}
    rng = rng or np.random.default_rng(2026)
    means = rng.choice(values, size=(n_boot, len(values)), replace=True).mean(axis=1)
    alpha = (1 - confidence) / 2
    return {
        'n': len(values), 'mean': float(values.mean()),
        'low': float(np.quantile(means, alpha)),
        'high': float(np.quantile(means, 1 - alpha)),
    }

seed_rows = per_seed_table(runs)
print_table(seed_rows)
for arm in ARMS:
    values = [row['final_binary_eval_return'] for row in seed_rows if row['arm'] == arm]
    ci = bootstrap_ci(values)
    if ci['n']:
        print(f"{arm}: final binary return = {ci['mean']:.3f} [{ci['low']:.3f}, {ci['high']:.3f}], n={ci['n']}")

In [ ]:
def held_out_table(runs: list[dict]) -> list[dict]:
    rows = []
    for run in runs:
        for distribution in TEST_DISTRIBUTIONS:
            result = run.get('evaluations', {}).get(distribution, {})
            value = result.get('binary_episode_score_mean')
            if value is not None:
                rows.append({
                    'arm': run.get('arm'), 'seed': run.get('seed'),
                    'distribution': distribution, 'binary_episode_score_mean': float(value),
                })
    return rows


def arm_summary(rows: list[dict], distribution: str = 'canonical') -> list[dict]:
    output = []
    for arm in ARMS:
        values = [
            row['binary_episode_score_mean'] for row in rows
            if row['arm'] == arm and row['distribution'] == distribution
        ]
        ci = bootstrap_ci(values)
        if ci['n']:
            output.append({
                'arm': arm, 'distribution': distribution, 'n': ci['n'],
                'mean': ci['mean'], 'ci_low': ci['low'], 'ci_high': ci['high'],
            })
    return output


def bootstrap_difference(rows: list[dict], arm_a: str, arm_b: str, distribution='canonical', n_boot=10_000) -> dict:
    a = np.asarray([row['binary_episode_score_mean'] for row in rows if row['arm'] == arm_a and row['distribution'] == distribution])
    b = np.asarray([row['binary_episode_score_mean'] for row in rows if row['arm'] == arm_b and row['distribution'] == distribution])
    if not len(a) or not len(b):
        return {'comparison': f'{arm_b} - {arm_a}', 'n_a': len(a), 'n_b': len(b), 'mean_difference': math.nan, 'low': math.nan, 'high': math.nan}
    rng = np.random.default_rng(2026)
    samples = rng.choice(b, (n_boot, len(b)), replace=True).mean(axis=1) - rng.choice(a, (n_boot, len(a)), replace=True).mean(axis=1)
    return {
        'comparison': f'{arm_b} - {arm_a}', 'n_a': len(a), 'n_b': len(b),
        'mean_difference': float(b.mean() - a.mean()),
        'low': float(np.quantile(samples, .025)), 'high': float(np.quantile(samples, .975)),
    }

held_out_rows = held_out_table(runs)
for distribution in TEST_DISTRIBUTIONS:
    print(f'\n{distribution.upper()}')
    print_table(arm_summary(held_out_rows, distribution))
primary_difference = bootstrap_difference(held_out_rows, 'A_sparse_fixed', 'D_dense_curriculum')
print('\nPrimary A-vs-D:', primary_difference)

## 7. Reward-mismatch check

Với B/D, so sánh final `collector_return` và binary canonical score. Gap lớn hoặc collector curve tăng trong khi binary curve đứng yên là tín hiệu cần xem trajectory/event histogram; đây là heuristic, không phải proof.

In [ ]:
def reward_mismatch_check(runs: list[dict], gap_threshold: float = 1.0) -> list[dict]:
    rows = []
    for run in runs:
        arm = run.get('arm')
        if arm not in {'B_dense_fixed', 'D_dense_curriculum', 'E_dense_broad'}:
            continue
        collector = final_metric(run, 'collector_return')
        binary_curve = final_metric(run, 'binary_eval_return')
        canonical = run.get('evaluations', {}).get('canonical', {}).get('binary_episode_score_mean', binary_curve)
        gap = collector - float(canonical) if np.isfinite(collector) and canonical is not None else math.nan
        reasons = []
        if np.isfinite(gap) and gap > gap_threshold:
            reasons.append(f'collector-binary gap > {gap_threshold}')
        if np.isfinite(collector) and collector > 0 and (canonical is None or float(canonical) <= 0):
            reasons.append('collector reward dương nhưng binary success bằng 0')
        rows.append({
            'run_id': run.get('run_id'), 'arm': arm, 'seed': run.get('seed'),
            'collector_final': collector, 'binary_canonical': canonical, 'gap': gap,
            'verdict': 'CHECK' if reasons else 'OK', 'reason': '; '.join(reasons) or 'không có cờ heuristic',
        })
    return rows

mismatch_flags = reward_mismatch_check(runs)
print_table(mismatch_flags)
if any(row['verdict'] == 'CHECK' for row in mismatch_flags):
    print('Không viết positive claim trước khi xem event histogram và trajectories của các run CHECK.')

## 8. Journal và go/no-go

Sinh Markdown dựa trên A–D/T0–T2. Go/no-go yêu cầu đủ seed, binary effect thực dụng và không có reward-mismatch flag; không dùng shaped return để quyết định.

In [ ]:
def markdown_journal(runs, held_out_rows, primary_difference, flags) -> str:
    stamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')
    lines = [
        '# SumToThree — curriculum + reward result note', '',
        f'- Generated: {stamp}', f'- Arm/seed records: {len(runs)}',
        '- Primary metric: canonical binary episode score', '',
        '## Primary A vs D',
        f"- Mean difference D-A: {primary_difference['mean_difference']:.3f}",
        f"- Bootstrap 95% CI: [{primary_difference['low']:.3f}, {primary_difference['high']:.3f}]",
        f"- n(A)={primary_difference['n_a']}, n(D)={primary_difference['n_b']}", '',
        '## T0/T1/T2 summaries',
    ]
    for distribution in TEST_DISTRIBUTIONS:
        lines.append(f'### {distribution}')
        for row in arm_summary(held_out_rows, distribution):
            lines.append(f"- {row['arm']}: {row['mean']:.3f} [{row['ci_low']:.3f}, {row['ci_high']:.3f}], n={row['n']}")
    checks = [row for row in flags if row['verdict'] == 'CHECK']
    lines += ['', '## Reward mismatch']
    lines += [f"- {row['run_id']}: {row['reason']}" for row in checks] or ['- No automatic flag; manual trajectory review still required.']
    lines += ['', '## Interpretation', '- Observation:', '- Alternative explanations:', '- Limitations:', '- Decision: supports | does_not_support | inconclusive', '- Next experiment:']
    return '\n'.join(lines)


def go_no_go(primary_difference: dict, flags: list[dict], min_seeds=3, practical_effect=0.5):
    reasons = []
    if primary_difference['n_a'] < min_seeds or primary_difference['n_b'] < min_seeds:
        reasons.append(f'Cần ít nhất {min_seeds} seeds cho A và D.')
    if not np.isfinite(primary_difference['mean_difference']) or primary_difference['mean_difference'] < practical_effect:
        reasons.append(f'D-A chưa đạt practical threshold {practical_effect} success/episode.')
    if not np.isfinite(primary_difference['low']) or primary_difference['low'] <= 0:
        reasons.append('Bootstrap CI của D-A chưa hoàn toàn dương.')
    if any(row['verdict'] == 'CHECK' for row in flags):
        reasons.append('Còn reward-mismatch flags.')
    return ('GO confirmation seeds', ['Effect đủ lớn; tiếp tục seeds 3/4 và manual review.']) if not reasons else ('NO-GO / INCONCLUSIVE', reasons)

journal = markdown_journal(runs, held_out_rows, primary_difference, mismatch_flags)
decision, reasons = go_no_go(primary_difference, mismatch_flags)
print(journal)
print('\n## Decision:', decision)
for reason in reasons:
    print('-', reason)

## Nộp lab

1. Tracker/manifest JSON và exact command cho từng arm/seed.
2. Random-policy diagnostic thật cho canonical/local/broad.
3. Binary learning curve + AUC theo seed cho A–D.
4. T0/T1/T2 binary evaluation table.
5. Bootstrap A-vs-D difference và reward-mismatch check.
6. Journal Markdown với observation, alternative explanations, limitations và quyết định go/no-go.

Synthetic demo trong notebook chỉ kiểm tra pipeline; tuyệt đối không dùng làm result.